# Academic Research Workflow: Full Analysis Pipeline

This notebook is designed for **graduate students and researchers** who need
publication-quality beta estimation with full diagnostics, benchmark comparisons,
and validation checks.

We will walk through the complete research pipeline:

1. Research-grade estimation with tuned hyperparameters
2. Inspecting the fitted model
3. Evaluation metrics
4. Benchmark comparison against rolling OLS
5. Lookahead bias validation
6. Exporting results for publication

In [ ]:
!pip install grubeta[full] -q

## Step 1: Research-Grade Estimation

The `preset="research"` option configures the model for maximum accuracy:
longer training, finer learning rate schedule, and full convergence checks.

In [ ]:
from grubeta import estimate_beta

result = estimate_beta("AAPL", "SPY", preset="research")
print(result["summary"])

## Step 2: Access the Fitted Model

The result dictionary exposes the underlying model and its configuration
for reproducibility and inspection.

In [ ]:
# Inspect the model and its configuration
model = result["model"]
print("Model configuration:")
print(f"  Lookback:            {model.config.lookback}")
print(f"  GRU units:           {model.config.gru_units}")
print(f"  Lambda beta:         {model.config.lambda_beta}")
print(f"  Lambda alpha:        {model.config.lambda_alpha}")
print(f"  Lambda alpha smooth: {model.config.lambda_alpha_smooth}")
print(f"  Walk-forward step:   {model.config.wf_step_size}")

print("\nResults DataFrame shape:", result["results"].shape)
print("\nResults columns:", list(result["results"].columns))
print("\nResults summary:")
print(result["results"].describe())

## Step 3: Evaluation Metrics

Use `BetaEvaluator` to compute standard evaluation metrics for the
estimated beta series, including in-sample fit and stability diagnostics.

In [ ]:
from grubeta.evaluation import BetaEvaluator

evaluator = BetaEvaluator()
metrics = evaluator.evaluate(
    betas=result["results"]["beta"].values,
    stock_returns=result["results"]["stock_return"].values,
    market_returns=result["results"]["market_return"].values,
    alphas=result["results"]["alpha"].values,
    name="AAPL_research"
)

print("Evaluation Metrics:")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"  {key:25s}: {value:.4f}")
    else:
        print(f"  {key:25s}: {value}")

## Step 4: Benchmark Comparison

Compare the GRU-estimated beta against a standard rolling OLS baseline.
This is essential for demonstrating that the neural approach adds value
over traditional methods.

In [ ]:
from grubeta.utils import rolling_ols_beta
import pandas as pd

# Compute rolling OLS beta as a benchmark
ols_beta = rolling_ols_beta(
    result["results"]["stock_return"].values,
    result["results"]["market_return"].values,
    window=252  # 1-year rolling window
)

# Side-by-side comparison
comparison = pd.DataFrame({
    "GRU Beta": result["results"]["beta"].values,
    "Rolling OLS Beta": ols_beta
}).dropna()

print("Comparison statistics:")
print(comparison.describe())
print(f"\nCorrelation between GRU and OLS beta: {comparison.corr().iloc[0, 1]:.4f}")

# Plot comparison
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(comparison.index, comparison["GRU Beta"], label="GRU Dynamic Beta", linewidth=1.5)
ax.plot(comparison.index, comparison["Rolling OLS Beta"], label="Rolling OLS (252d)", alpha=0.7)
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.4)
ax.set_title("GRU vs Rolling OLS Beta \u2014 AAPL")
ax.set_ylabel("Beta (\u03b2)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5: Lookahead Bias Validation

A critical check for any time-series model: verify that the estimated beta
at time *t* does not use any information from time *t+1* or later.

The `validate_no_lookahead` function runs an expanding-window test to confirm
that predictions are stable when future data is excluded.

In [ ]:
from grubeta.utils import validate_no_lookahead

passed = validate_no_lookahead(
    betas=result["results"]["beta"].values,
    returns=result["results"]["stock_return"].values,
    market_returns=result["results"]["market_return"].values,
    initial_size=result["model"].config.initial_train_size,
)

print(f"\nLookahead bias test: {'PASSED \u2713' if passed else 'FAILED \u2717'}")

In [ ]:
# Save beta series to CSV
result["results"].to_csv("aapl_dynamic_beta_results.csv", index=False)
print("Results saved to aapl_dynamic_beta_results.csv")

# Save model for future use
result["model"].save("aapl_research_model")
print("Model saved to aapl_research_model/")

# Generate publication-ready figure
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

valid = result["results"].dropna(subset=["beta"])
dates = valid["date"] if "date" in valid.columns else valid.index

# Panel 1: Beta trajectory
axes[0].plot(dates, valid["beta"], color="#2c3e50", linewidth=1.5)
axes[0].axhline(1.0, color="#e74c3c", linestyle="--", alpha=0.4)
axes[0].set_ylabel("Beta (\u03b2)")
axes[0].set_title("AAPL Dynamic Beta \u2014 Research Preset")
axes[0].grid(True, alpha=0.2)

# Panel 2: Alpha trajectory
axes[1].plot(dates, valid["alpha"], color="#27ae60", linewidth=1, alpha=0.8)
axes[1].axhline(0.0, color="gray", linestyle="--", alpha=0.4)
axes[1].set_ylabel("Alpha (\u03b1)")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("aapl_research_beta_alpha.png", dpi=300, bbox_inches="tight")
plt.show()
print("Publication figure saved to aapl_research_beta_alpha.png")